# Online Retail II: Sales Analysis with DuckDB SQL

In this project I use SQL to analyze two years of transactions from a UK-based online retailer. First I load and check the raw data, then I prepare it for analysis, and finally I answer business questions about revenue, countries, customers, products, order times and cancellations.

**Dataset:** [Online Retail II](https://archive.ics.uci.edu/dataset/502/online+retail+ii) from the UC Irvine Machine Learning Repository. It contains about one million invoice lines from 1 December 2009 to 9 December 2011. The company mostly sells giftware, and many of its customers are wholesalers. Prices are in pounds sterling £. Licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

**Tools:** Jupyter Notebook in VS Code, DuckDB SQL, Python, pandas to display the results

## Key findings

- **Strong seasonality:** Net revenue rises strongly from September to November. November 2011 was the best month with £1.43 million.
- **Light growth:** From January to November, net revenue in 2011 was about 2% higher than in 2010.
- **Home market:** The United Kingdom generates 85% of the net revenue. Customers in the Netherlands place the largest orders, averaging £2,545 compared to £466 in the UK.
- **Loyal customers:** About 72% of the customers with a customer ID ordered more than once, and they generate about 97% of the revenue from these customers. The top 10 customers alone generate 16%.
- **Order times:** Most orders come in from Tuesday to Thursday around lunchtime. There are almost no orders on Saturdays.
- **Cancellations:** 4.2% of all sold units are cancelled. About half of them are orders that were cancelled on the same day. Without these, the rate is 2.2%.

## Contents

1. Setup
2. Loading the data
3. Data exploration and quality checks
4. Preparing the data for analysis
5. Business questions
6. Recommendations and limitations

## How to run this notebook

1. Download `online_retail_II.xlsx` from the UCI link above and save it in the `data` folder.
2. Install the packages from `requirements.txt` in a virtual environment as described in the README.
3. Run all cells. The notebook creates the database file `Online Retail II_Dataset.duckdb` in the `data` folder automatically.



## 1. Setup

I use DuckDB as the database and pandas only to show the query results as tables. The helper function `q()` runs a SQL query and returns the result as a pandas DataFrame. The `excel` extension allows DuckDB to read the Excel file directly.

In [1]:
import duckdb
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

con = duckdb.connect("../data/Online Retail II_Dataset.duckdb")

con.execute("INSTALL excel")
con.execute("LOAD excel")

def q(sql):
    return con.sql(sql).df()

## 2. Loading the data

The Excel file has two sheets: `Year 2009-2010` and `Year 2010-2011`. I load each sheet into its own table and check it before I combine them.

While loading, I clean the columns:
- I read all columns as text and then convert them, so that DuckDB does not have to guess the data types.
- `InvoiceDate` is stored as an Excel date, which shows the number of days since December 30, 1899. I convert it to a timestamp and round it to whole seconds.
- `Customer ID` is converted to a whole number and then to text, because it is an identifier and not a value to calculate with. I also rename it to `Customer_ID`.
- `Description` is trimmed to remove spaces at the beginning and at the end.

### 2.1 Sheet 2009–2010

In [2]:
con.execute("""
    CREATE OR REPLACE TABLE sheet_2009_2010 AS
    SELECT
        Invoice,
        StockCode,
        trim(Description) AS Description,
        CAST(Quantity AS BIGINT) AS Quantity,
        TIMESTAMP '1899-12-30' + to_seconds(round(CAST(InvoiceDate AS DOUBLE) * 86400)) AS InvoiceDate,
        CAST(Price AS DOUBLE) AS Price,
        CAST(CAST("Customer ID" AS BIGINT) AS VARCHAR) AS Customer_ID,
        Country
    FROM read_xlsx('../data/online_retail_II.xlsx',
        sheet = 'Year 2009-2010',
        all_varchar = true)
""");

A first look at the rows, the data types and the number of rows:

In [3]:
q("FROM sheet_2009_2010 LIMIT 5")

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom


In [4]:
q("DESCRIBE FROM sheet_2009_2010")

,column_name,column_type,null,key,default,extra
0,Invoice,VARCHAR,YES,None,None,None
1,StockCode,VARCHAR,YES,None,None,None
2,Description,VARCHAR,YES,None,None,None
3,Quantity,BIGINT,YES,None,None,None
4,InvoiceDate,TIMESTAMP,YES,None,None,None
5,Price,DOUBLE,YES,None,None,None
6,Customer_ID,VARCHAR,YES,None,None,None
7,Country,VARCHAR,YES,None,None,None


In [5]:
q("""
    SELECT COUNT(*) AS number_of_rows
    FROM sheet_2009_2010
""")

,number_of_rows
0,525461


#### Duplicated rows

How many rows are exact copies of another row, and how much revenue do they represent?

In [6]:
q("""
    WITH unique_rows AS (
        SELECT DISTINCT *
        FROM sheet_2009_2010
    )
    SELECT
        (SELECT COUNT(*) FROM sheet_2009_2010) - (SELECT COUNT(*) FROM unique_rows) AS duplicated_rows
""")

,duplicated_rows
0,6865


In [7]:
q("""
    SELECT
        *,
        COUNT(*) AS occurrences
    FROM sheet_2009_2010
    GROUP BY ALL
    HAVING COUNT(*) > 1
    ORDER BY occurrences DESC
    LIMIT 5
""")

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country,occurrences
0,502660,17021,NAMASTE SWAGAT INCENSE,6,2010-03-25 17:18:00,0.30,13187,United Kingdom,6
1,534219,35953,FOLKART STAR CHRISTMAS DECORATIONS,1,2010-11-21 16:13:00,1.25,13230,United Kingdom,6
2,496431,84826,ASSTD DESIGN 3D PAPER STICKERS,1,2010-02-01 12:30:00,0.85,16415,United Kingdom,6
3,525065,20894,HANGING BAUBLE T-LIGHT HOLDER LARGE,1,2010-10-03 14:28:00,2.95,16799,United Kingdom,6
4,520570,21755,LOVE BUILDING BLOCK WORD,1,2010-08-26 20:07:00,5.95,16549,United Kingdom,5


In [8]:
q("""
    WITH all_orders AS (
        SELECT
            SUM(Quantity * Price) AS net_revenue
        FROM sheet_2009_2010
    ),
    grouped_orders AS (
        SELECT
            *,
            COUNT(*) AS occurrences
        FROM sheet_2009_2010
        GROUP BY ALL
        HAVING COUNT(*) > 1
    ),
    previous_with_duplicated_rows AS (
        SELECT
            SUM(Quantity * Price) AS duplicated_rows_net_revenue
        FROM grouped_orders
    )
    SELECT
        ROUND(100.0
            * (FROM previous_with_duplicated_rows)
            / (FROM all_orders)
            ,2) AS percentage_revenue_from_duplicated_rows
""")

,percentage_revenue_from_duplicated_rows
0,0.34


**Finding:** 6,865 rows are exact duplicates. It is not clear whether they are input errors or real repeated purchases. Because they represent only about 0.34% of the revenue, I keep them.

### 2.2 Sheet 2010–2011

I load the second sheet in the same way and run the same checks.

In [9]:
con.execute("""
    CREATE OR REPLACE TABLE sheet_2010_2011 AS
    SELECT
        Invoice,
        StockCode,
        trim(Description) AS Description,
        CAST(Quantity AS BIGINT) AS Quantity,
        TIMESTAMP '1899-12-30' + to_seconds(round(CAST(InvoiceDate AS DOUBLE) * 86400)) AS InvoiceDate,
        CAST(Price AS DOUBLE) AS Price,
        CAST(CAST("Customer ID" AS BIGINT) AS VARCHAR) AS Customer_ID,
        Country
    FROM read_xlsx('../data/online_retail_II.xlsx',
        sheet = 'Year 2010-2011',
        all_varchar = true)
""");

In [10]:
q("FROM sheet_2010_2011 LIMIT 5")

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


In [11]:
q("DESCRIBE FROM sheet_2010_2011")

,column_name,column_type,null,key,default,extra
0,Invoice,VARCHAR,YES,None,None,None
1,StockCode,VARCHAR,YES,None,None,None
2,Description,VARCHAR,YES,None,None,None
3,Quantity,BIGINT,YES,None,None,None
4,InvoiceDate,TIMESTAMP,YES,None,None,None
5,Price,DOUBLE,YES,None,None,None
6,Customer_ID,VARCHAR,YES,None,None,None
7,Country,VARCHAR,YES,None,None,None


In [12]:
q("""
    SELECT
        COUNT(*) AS number_of_rows
    FROM sheet_2010_2011
""")

,number_of_rows
0,541910


In [13]:
q("""
    WITH unique_rows AS (
        SELECT DISTINCT *
        FROM sheet_2010_2011
    )
    SELECT
        (SELECT COUNT(*) FROM sheet_2010_2011) - (SELECT COUNT(*) FROM unique_rows) AS duplicated_rows
""")

,duplicated_rows
0,5268


In [14]:
q("""
    SELECT
        *,
        COUNT(*) AS occurrences
    FROM sheet_2010_2011
    GROUP BY ALL
    HAVING COUNT(*) > 1
    ORDER BY occurrences DESC
    LIMIT 5
""")

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country,occurrences
0,555524,22698,PINK REGENCY TEACUP AND SAUCER,1,2011-06-05 11:37:00,2.95,16923,United Kingdom,20
1,555524,22697,GREEN REGENCY TEACUP AND SAUCER,1,2011-06-05 11:37:00,2.95,16923,United Kingdom,12
2,572861,22775,PURPLE DRAWERKNOB ACRYLIC EDWARDIAN,12,2011-10-26 12:46:00,1.25,14102,United Kingdom,8
3,578289,23395,BELLE JARDINIERE CUSHION COVER,1,2011-11-23 14:07:00,3.75,17841,United Kingdom,6
4,572344,M,Manual,48,2011-10-24 10:43:00,1.50,14607,United Kingdom,6


In [15]:
q("""
    WITH all_orders AS (
        SELECT
            SUM(Quantity * Price) AS net_revenue
        FROM sheet_2010_2011
    ),
    grouped_orders AS (
        SELECT
            *,
            COUNT(*) AS occurrences
        FROM sheet_2010_2011
        GROUP BY ALL
        HAVING COUNT(*) > 1
    ),
    previous_with_duplicated_rows AS (
        SELECT
            SUM(Quantity * Price) AS duplicated_rows_net_revenue
        FROM grouped_orders
    )
    SELECT
        ROUND(100.0
            * (FROM previous_with_duplicated_rows)
            / (FROM all_orders)
            ,2) AS percentage_revenue_from_duplicated_rows
""")

,percentage_revenue_from_duplicated_rows
0,0.21


**Finding:** 5,268 rows are exact duplicates. They represent only about 0.21% of the revenue, so I keep them as well.

### 2.3 Combining the two sheets

The two sheets overlap: the first sheet ends on 9 December 2010, but the second sheet already starts on 1 December 2010. I check how many rows appear in both sheets:

In [16]:
q("""
    SELECT
        COUNT(*) AS overlapping_rows
    FROM (
        FROM sheet_2009_2010
        INTERSECT
        FROM sheet_2010_2011
    )
""")

,overlapping_rows
0,22202


In [17]:
q("""
    FROM sheet_2009_2010
    INTERSECT
    FROM sheet_2010_2011
    ORDER BY Invoice
    LIMIT 5
""")

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country
0,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
1,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
2,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
3,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom
4,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850,United Kingdom


**Finding:** 22,202 distinct rows appear in both sheets. To avoid counting them twice, I take all rows of the first sheet and add only rows of the second sheet that are not in the first sheet.

In [18]:
con.execute("""
    CREATE OR REPLACE TABLE combined_sheets AS
        WITH sheet_2010_2011_without_overlap_sheet_2009_2010 AS (
            FROM sheet_2010_2011
            EXCEPT ALL
            FROM sheet_2009_2010
        )
        FROM sheet_2009_2010
        UNION ALL
        FROM sheet_2010_2011_without_overlap_sheet_2009_2010
""");

## 3. Data exploration and quality checks

In this section I check whether the combined data is plausible and look for problems that could distort the results. For each problem I decide how to handle it.

### 3.1 Date range

In [19]:
q("""
    FROM combined_sheets
    ORDER BY InvoiceDate
    LIMIT 5
""")

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom


In [20]:
q("""
    FROM combined_sheets
    ORDER BY InvoiceDate DESC
    LIMIT 5
""")

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country
0,581587,POST,POSTAGE,1,2011-12-09 12:50:00,18.00,12680,France
1,581587,22631,CIRCUS PARADE LUNCH BOX,12,2011-12-09 12:50:00,1.95,12680,France
2,581587,22726,ALARM CLOCK BAKELIKE GREEN,4,2011-12-09 12:50:00,3.75,12680,France
3,581587,22367,CHILDRENS APRON SPACEBOY DESIGN,8,2011-12-09 12:50:00,1.95,12680,France
4,581587,22555,PLASTERS IN TIN STRONGMAN,12,2011-12-09 12:50:00,1.65,12680,France


**Finding:** The data covers the period from 1 December 2009 to 9 December 2011. This means that **December 2011 is incomplete**, which I have to keep in mind for all monthly analyses.

### 3.2 What does one row represent?

In [21]:
q("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT Invoice) AS unique_Invoice,
        COUNT(DISTINCT StockCode) AS unique_StockCode,
        COUNT(DISTINCT Customer_ID) AS unique_Customer_ID
    FROM combined_sheets
""")

,total_rows,unique_Invoice,unique_StockCode,unique_Customer_ID
0,1044848,53628,5305,5942


The combined table has 1,044,848 rows, 53,628 invoices, 5,305 stock codes and 5,942 customer IDs. The next three queries show how invoices, products and customers are related:

In [22]:
q("""
    WITH invoice_counts AS (
        SELECT
            *,
            COUNT(*) OVER (PARTITION BY Invoice) AS orders_per_invoice
        FROM combined_sheets
    )
    SELECT * EXCLUDE (orders_per_invoice)
    FROM invoice_counts
    WHERE orders_per_invoice > 1
    ORDER BY Invoice
    LIMIT 20
""")

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom
4,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085,United Kingdom
5,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
6,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085,United Kingdom
7,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
8,489435,22195,HEART MEASURING SPOONS LARGE,24,2009-12-01 07:46:00,1.65,13085,United Kingdom
9,489435,22353,LUNCHBOX WITH CUTLERY FAIRY CAKES,12,2009-12-01 07:46:00,2.55,13085,United Kingdom


In [23]:
q("""
    WITH product_counts AS (
        SELECT
            *,
            COUNT(*) OVER (PARTITION BY StockCode) AS orders_per_product
        FROM combined_sheets
    )
    SELECT * EXCLUDE (orders_per_product)
    FROM product_counts
    WHERE orders_per_product > 1
    ORDER BY StockCode
    LIMIT 20
""")

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country
0,501912,10002,INFLATABLE POLITICAL GLOBE,60,2010-03-22 09:19:00,0.85,17552,United Kingdom
1,503160,10002,INFLATABLE POLITICAL GLOBE,24,2010-03-30 13:00:00,0.72,12931,United Kingdom
2,502751,10002,INFLATABLE POLITICAL GLOBE,1,2010-03-26 13:13:00,0.85,17290,United Kingdom
3,503956,10002,INFLATABLE POLITICAL GLOBE,4,2010-04-08 16:45:00,0.85,13443,United Kingdom
4,501267,10002,INFLATABLE POLITICAL GLOBE,1,2010-03-15 12:54:00,0.85,17085,United Kingdom
5,503503,10002,INFLATABLE POLITICAL GLOBE,24,2010-04-01 12:30:00,0.85,12583,France
6,502721,10002,INFLATABLE POLITICAL GLOBE,12,2010-03-26 11:21:00,0.85,15789,United Kingdom
7,503724,10002,INFLATABLE POLITICAL GLOBE,12,2010-04-06 16:24:00,0.85,13050,United Kingdom
8,503907,10002,INFLATABLE POLITICAL GLOBE,200,2010-04-08 12:23:00,0.72,18251,United Kingdom
9,C503527,10002,INFLATABLE POLITICAL GLOBE,-200,2010-04-01 13:13:00,0.72,16019,United Kingdom


In [24]:
q("""
    WITH customer_counts AS (
        SELECT
            *,
            COUNT(*) OVER (PARTITION BY Customer_ID) AS orders_per_customer
        FROM combined_sheets
    )
    SELECT * EXCLUDE (orders_per_customer)
    FROM customer_counts
    WHERE orders_per_customer > 1
    ORDER BY Customer_ID
    LIMIT 20
""")

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country
0,513774,22688,DOORMAT PEACE ON EARTH BLUE,1,2010-06-28 13:53:00,7.49,12346,United Kingdom
1,513774,21524,DOORMAT SPOTTY HOME SWEET HOME,1,2010-06-28 13:53:00,7.49,12346,United Kingdom
2,513774,22691,DOORMAT WELCOME SUNRISE,1,2010-06-28 13:53:00,7.49,12346,United Kingdom
3,513774,22690,DOORMAT HOME SWEET HOME BLUE,1,2010-06-28 13:53:00,7.49,12346,United Kingdom
4,513774,48185,DOORMAT FAIRY CAKE,1,2010-06-28 13:53:00,7.49,12346,United Kingdom
5,513774,48111,DOORMAT 3 SMILEY CATS,1,2010-06-28 13:53:00,7.49,12346,United Kingdom
6,513774,48188,DOORMAT WELCOME PUPPIES,1,2010-06-28 13:53:00,7.49,12346,United Kingdom
7,513774,22689,DOORMAT MERRY CHRISTMAS RED,1,2010-06-28 13:53:00,7.49,12346,United Kingdom
8,513774,20685,DOORMAT RED SPOT,1,2010-06-28 13:53:00,7.49,12346,United Kingdom
9,513774,48138,DOORMAT UNION FLAG,1,2010-06-28 13:53:00,7.49,12346,United Kingdom


**Finding:** One row is one product line on an invoice. An invoice contains multiple products, a product appears on many invoices, and a customer can have many invoices.

### 3.3 Missing values

In [25]:
q("""
    SELECT
        COUNT(*) FILTER (WHERE Invoice IS NULL) AS invoice_nulls,
        COUNT(*) FILTER (WHERE StockCode IS NULL) AS stockcode_nulls,
        COUNT(*) FILTER (WHERE Description IS NULL) AS description_nulls,
        COUNT(*) FILTER (WHERE Quantity IS NULL) AS quantity_nulls,
        COUNT(*) FILTER (WHERE InvoiceDate IS NULL) AS invoicedate_nulls,
        COUNT(*) FILTER (WHERE Price IS NULL) AS price_nulls,
        COUNT(*) FILTER (WHERE Customer_ID IS NULL) AS customer_id_nulls,
        COUNT(*) FILTER (WHERE Country IS NULL) AS country_nulls
    FROM combined_sheets
""")

,invoice_nulls,stockcode_nulls,description_nulls,quantity_nulls,invoicedate_nulls,price_nulls,customer_id_nulls,country_nulls
0,0,0,4275,0,0,0,235287,0


Only `Description` with 4,275 rows and `Customer_ID` with 235,287 rows have missing values. I identify products by `StockCode`, so missing descriptions are not a problem. Missing customer IDs are more important, so I check how many invoices and how much revenue they influence:

In [26]:
q("""
    SELECT
        COUNT(DISTINCT Invoice) FILTER (WHERE Customer_ID IS NULL)
            AS distinct_invoices_without_customer_id,
        COUNT(DISTINCT Invoice) AS total_invoices,
        ROUND(100.0
            * COUNT(DISTINCT Invoice) FILTER (WHERE Customer_ID IS NULL)
            / COUNT(DISTINCT Invoice),
            2) AS percent_distinct_invoices_without_customer_id
    FROM combined_sheets

""")

,distinct_invoices_without_customer_id,total_invoices,percent_distinct_invoices_without_customer_id
0,8752,53628,16.32


In [27]:
q("""
    SELECT
        SUM(Quantity * Price) FILTER (WHERE Customer_ID IS NULL) 
            AS revenue_without_customer_id,
        SUM(Quantity * Price) AS total_revenue,
        ROUND(100.0
            * SUM(Quantity * Price) FILTER (WHERE Customer_ID IS NULL)
            / SUM(Quantity * Price)
            , 2) AS percent_revenue_without_customer_id
    FROM combined_sheets
""")

,revenue_without_customer_id,total_revenue,percent_revenue_without_customer_id
0,2566093.08,1.890976e+07,13.57


**Finding:** Invoices without a customer ID make up 16.3% of all invoices and 13.6% of the revenue. This is too much to delete, so I keep these rows for all revenue questions. I only exclude them when a question is about individual customers.

### 3.4 Cancellations, negative values and zero values

In [28]:
q("""
    SELECT
        COUNT(*) FILTER (WHERE Invoice LIKE 'C%') AS count_cancellations,
        COUNT(*) FILTER (WHERE Quantity < 0) AS count_negative_quantity,
        COUNT(*) FILTER (WHERE Quantity = 0) AS count_zero_quantity,
        COUNT(*) FILTER (WHERE Price < 0) AS count_negative_price,
        COUNT(*) FILTER (WHERE Price = 0) AS count_zero_price
    FROM combined_sheets
""")

,count_cancellations,count_negative_quantity,count_zero_quantity,count_negative_price,count_zero_price
0,19165,22557,0,5,6024


**Finding:** There are 19,165 cancellation lines and 22,557 lines with a negative quantity. No line has a quantity of 0, but 6,024 lines have a price of 0 and 5 lines have a negative price. Prices of 0 or below and negative quantities on normal invoices are not real sales, so I exclude them later in section 4.

### 3.5 Stock codes that are not products

Normal stock codes start with a number. This query lists all stock codes that do not:

In [29]:
q("""
    SELECT
        StockCode,
        MODE(Description) AS most_values,
        COUNT(*) AS count_rows
    FROM combined_sheets
    WHERE NOT regexp_matches(StockCode, '^[0-9]+')
    GROUP BY StockCode
    ORDER BY count_rows DESC
""")

,StockCode,most_values,count_rows
0,POST,POSTAGE,2086
1,DOT,DOTCOM POSTAGE,1425
2,M,Manual,1398
3,C2,CARRIAGE,277
4,D,Discount,173
5,S,SAMPLES,102
6,BANK CHARGES,Bank Charges,100
7,ADJUST,Adjustment by john on 26/01/2010 16,67
8,AMAZONFEE,AMAZON FEE,36
9,DCGS0058,MISO PRETTY GUM,31


**Finding:** Some codes are not products but postage, fees, manual adjustments, gift vouchers or test entries. I add the column `non_product_type` to mark these rows, so that I can exclude them from the analysis. Codes like `DCGS...`, `PADS` and `SP1002` look like real products, so I keep them.

In [30]:
con.execute("""
    ALTER TABLE combined_sheets
    ADD COLUMN IF NOT EXISTS non_product_type VARCHAR;

    UPDATE combined_sheets
    SET non_product_type = CASE
        WHEN UPPER(StockCode) IN ('POST', 'DOT', 'C2', 'C3') THEN 'shipping'
        WHEN UPPER(StockCode) IN ('BANK CHARGES', 'AMAZONFEE', 'CRUK') THEN 'fee commission'
        WHEN UPPER(StockCode) IN ('M', 'D', 'S', 'B', 'ADJUST', 'ADJUST2') THEN 'manual adjustment'
        WHEN UPPER(StockCode) LIKE 'GIFT%' THEN 'gift_voucher'
        WHEN UPPER(StockCode) LIKE 'TEST%' THEN 'test'
        ELSE NULL
    END;
""");

In [31]:
q("""
    SELECT
        COUNT(*) AS total_rows,
        non_product_type
    FROM combined_sheets
    GROUP BY non_product_type
    ORDER BY total_rows DESC
""")

,total_rows,non_product_type
0,1039035,NaN
1,3789,shipping
2,1754,manual adjustment
3,152,fee commission
4,101,gift_voucher
5,17,test


In [32]:
q("""
    SELECT
        COUNT(*) FILTER (WHERE non_product_type IS NOT NULL) AS count_non_products,
        ROUND(100.0
            * COUNT(*) FILTER (WHERE non_product_type IS NOT NULL)
            / COUNT(*)
            , 2) AS percent_non_product_rows
    FROM combined_sheets
""")

,count_non_products,percent_non_product_rows
0,5813,0.56


0.56% of all rows are marked as non-product rows.

### 3.6 Invoices with and without a customer ID

Next, I compare invoices with and without a customer ID. For this comparison I only use real sales. That means no cancellations, no non-product rows, and only positive quantities and prices.

In [33]:
q("""
    WITH invoice_level AS (
        SELECT
            Invoice,
            CASE
                WHEN Customer_ID IS NOT NULL THEN true
                ELSE false
                END AS has_customer_id,
            COUNT(*) AS count_of_rows,
            SUM(Quantity) AS units,
            SUM(Quantity * Price) AS invoice_value
        FROM combined_sheets
        WHERE
            Invoice NOT LIKE 'C%'
            AND Quantity > 0
            AND Price > 0
            AND non_product_type IS NULL
        GROUP BY
            Invoice,
            has_customer_id
    )
    SELECT
        has_customer_id,
        COUNT(*) AS invoices,
        ROUND(AVG(count_of_rows), 2) AS average_rows,
        ROUND(AVG(
            units
            / count_of_rows)
            , 2) AS average_units_per_row,
        ROUND(MEDIAN(invoice_value), 2) AS median_invoice_value,
        ROUND(AVG(invoice_value), 2) AS avg_invoice_value
    FROM invoice_level
    GROUP BY has_customer_id
""")

,has_customer_id,invoices,average_rows,average_units_per_row,median_invoice_value,avg_invoice_value
0,False,2922,77.65,3.46,184.48,881.59
1,True,36594,21.54,33.72,303.10,467.97


**Finding:** Invoices without a customer ID have many more lines, averaging 78 compared with 22. However, they have far fewer units per line, averaging 3.5 compared with 34. Their median value is lower at £184 compared with £303. Their average value is higher, at £882 compared with £468, because a few of them are very large. One possible explanation is that these are orders placed without a customer account, while customers with an ID are often wholesalers who buy large quantities.

### 3.7 Stock codes with several descriptions

In [34]:
q("""
    SELECT
        StockCode,
        COUNT(DISTINCT Description) AS count_of_descriptions,
        list(DISTINCT Description) AS descriptions
    FROM combined_sheets
    WHERE
        non_product_type IS NULL
        AND Description IS NOT NULL
    GROUP BY StockCode
    HAVING COUNT(DISTINCT Description) > 1
    ORDER BY count_of_descriptions DESC
    LIMIT 20
""")

,StockCode,count_of_descriptions,descriptions
0,20713,9,"[Marked as 23343, wrongly coded 23343, wrongly marked 23343, wrongly coded-23343, JUMBO BAG OWLS, found, missing, Found, wrongly marked. 23343 in box]"
1,21181,7,"[adjustment, PLEASE ONE PERSON METAL SIGN, missing, check, on cargo order, PLEASE ONE PERSON METAL SIGN, dotcom]"
2,22423,7,"[broken, uneven bottom, damages, wonky bottom/broken, REGENCY CAKESTAND 3 TIER, smashed, damaged, faulty]"
3,23084,7,"[for online retail orders, Amazon, add stock to allocate online orders, temp adjustment, RABBIT NIGHT LIGHT, allocate stock for dotcom orders ta, website fixed]"
4,22734,7,"[amazon adjustment, SET OF 6 RIBBONS VINTAGE CHRISTMAS, FOUND, amazon sales, amendment, Carton qnty was 216 not 144 as stat, amazon]"
5,47566B,6,"[missing, correct previous adjustment, reverse previous adjustment, incorrectly credited C550456 see 47, TEA TIME PARTY BUNTING, stock credited from royal yacht inc]"
6,22719,6,"[GUMBALL COATHOOK., BLACK & WHITE, GUMBALL MONOCHROME COAT RACK, wrong code, sold as 22467, wrong barcode (22467), 22467]"
7,85175,6,"[CACTI T-LIGHT CANDLES, check, Amazon sold sets, dotcom sold sets, wrongly sold sets, ? sold as sets?]"
8,21830,6,"[ASSORTED CREEPY CRAWLIES, damaged, ?, OOPS ! adjustment, MERCHANT CHANDLER CREDIT ERROR, STO, sold as 1]"
9,23131,5,"[MISELTOE HEART WREATH WHITE, MISELTOE HEART WREATH CREAM, had been put aside, MISTLETOE HEART WREATH CREAM, ?]"


In [35]:
q("""
    WITH grouped_stock_codes AS (
        SELECT
            StockCode,
            COUNT(DISTINCT Description) AS count_of_different_descriptions
        FROM combined_sheets
        WHERE non_product_type IS NULL
        GROUP BY StockCode
    )
    SELECT
        COUNT(*) FILTER (WHERE count_of_different_descriptions > 1) AS stock_codes_with_different_descriptions,
        COUNT(*) AS count_of_rows,
        ROUND(100.0
            * stock_codes_with_different_descriptions
            / count_of_rows
        , 2) AS percentage_of_stock_codes_with_different_descriptions
    FROM grouped_stock_codes
""")

,stock_codes_with_different_descriptions,count_of_rows,percentage_of_stock_codes_with_different_descriptions
0,1212,5279,22.96


**Finding:** 23% of the stock codes have more than one description. Many of the additional descriptions are notes such as "damaged", "found" or "wrongly marked" instead of product names. Because all calculations are grouped by `StockCode`, this does not affect the numbers. To label the products in the results, I use the most frequent description of each stock code.

### 3.8 Countries

In [36]:
q("""
    SELECT DISTINCT Country
    FROM combined_sheets
    ORDER BY Country

""")

,Country
0,Australia
1,Austria
2,Bahrain
3,Belgium
4,Bermuda
5,Brazil
6,Canada
7,Channel Islands
8,Cyprus
9,Czech Republic


Most country names are clear, but a few need an explanation: `EIRE` is Ireland and `RSA` is South Africa. `European Community` and `Unspecified` are not real countries, so I check how often they appear:

In [37]:
q("""
    SELECT
        COUNT(DISTINCT Invoice)
            FILTER (WHERE Country = 'European Community') AS count_order_with_european_community,
        COUNT(DISTINCT Invoice)
            FILTER (WHERE Country = 'Unspecified') AS count_order_with_unspecified
    FROM combined_sheets
""")

,count_order_with_european_community,count_order_with_unspecified
0,5,28


**Finding:** Only 5 invoices are from "European Community" and 28 are "Unspecified". This is too few to affect the results, so I keep them unchanged.

### 3.9 Outliers in quantity and price

I look at the minimum, median, 99th percentile and maximum of quantity and price for real sales:

In [38]:
q("""
    SELECT
        MIN(Quantity) AS minimal_quantity,
        quantile_cont(Quantity, 0.5) AS median_quantity,
        quantile_cont(Quantity, 0.99) AS percentile_99_quantity,
        MAX(Quantity) AS maximal_quantity,
        MIN(Price) AS minimal_price,
        quantile_cont(Price, 0.5) AS median_price,
        quantile_cont(Price, 0.99) AS percentile_99_price,
        MAX(Price) AS maximal_price
    FROM combined_sheets
    WHERE
        Invoice NOT LIKE 'C%'
        AND Price > 0
        AND Quantity > 0
        AND non_product_type IS NULL
""")

,minimal_quantity,median_quantity,percentile_99_quantity,maximal_quantity,minimal_price,median_price,percentile_99_price,maximal_price
0,1,3.0,100.0,80995,0.001,2.1,16.95,1157.15


**Finding:** 99% of the lines have a quantity of 100 or less and a price of £16.95 or less, but the maximum quantity is 80,995. In section 5.6 I show that this order was cancelled on the same day. Instead of deleting such orders, I create a second view without same-day cancellations in section 4 and compare the results of both views.

## 4. Preparing the data for analysis

Based on the checks above, I add a column with the revenue of each line and create two views for the business questions.

In [39]:
con.execute("""
    ALTER TABLE combined_sheets ADD COLUMN IF NOT EXISTS line_total DOUBLE;

    UPDATE combined_sheets
    SET
        line_total = Quantity * Price;
""");

**`v_sales_lines`** is the main view for all business questions. It contains:
- only products,
- only rows with a price above 0,
- normal sales with a positive quantity and cancellations

Because cancellations have a negative quantity, `SUM(line_total)` returns the **net revenue**, i.e., sales minus cancellations.

In [40]:
con.execute("""
    CREATE OR REPLACE VIEW v_sales_lines AS
        SELECT *
        FROM combined_sheets
        WHERE
            non_product_type IS NULL
            AND Price > 0
            AND (
                (Invoice NOT LIKE 'C%' AND Quantity > 0)  -- sales
                OR (Invoice LIKE 'C%' AND Quantity < 0)   -- cancellations
            )
""");

**`v_sales_lines_clean`** shows how much same-day cancellations influence the results. It removes every sale for which the same customer cancelled the same product with the same quantity on the same day, together with the matching cancellation. These were most likely mistakes that were corrected right away. Rows without a customer ID cannot be matched, so they stay in the view.

In [41]:
con.execute("""
    CREATE OR REPLACE VIEW v_sales_lines_clean AS
        SELECT s.*
        FROM v_sales_lines s
        WHERE NOT EXISTS (
            -- a row of the same customer and product with the opposite quantity on the same day
            FROM v_sales_lines cancel_row
            WHERE
                cancel_row.Customer_ID = s.Customer_ID
                AND cancel_row.StockCode = s.StockCode
                AND cancel_row.Quantity = -s.Quantity
                AND CAST(cancel_row.InvoiceDate AS DATE) = CAST(s.InvoiceDate AS DATE)
        )
""");

## 5. Business questions

All questions use the view `v_sales_lines`. **Net revenue** therefore means sales minus cancellations, without postage and other non-product rows. Some questions also use `v_sales_lines_clean` to check how much same-day cancellations change the result.

Please keep in mind:
- December 2011 only contains data up to 9 December.
- Questions about individual customers only include rows with a customer ID.

### 5.1 Revenue over time

#### Q1. What was the net revenue per month?

In [42]:
q("""
    WITH revenue AS (
        SELECT
            year(InvoiceDate) AS year,
            month(InvoiceDate) AS month_number,
            monthname(InvoiceDate) AS month,
            SUM(line_total) AS net_revenue
        FROM v_sales_lines
        GROUP BY
            year(InvoiceDate),
            month(InvoiceDate),
            monthname(InvoiceDate)
    )
    SELECT
        year,
        month,
        ROUND(net_revenue, 2) AS net_revenue
    FROM revenue
    ORDER BY
        year,
        month_number
""")

,year,month,net_revenue
0,2009,December,781541.69
1,2010,January,606385.18
2,2010,February,527160.30
3,2010,March,754676.65
4,2010,April,638968.56
5,2010,May,608351.25
6,2010,June,672935.41
7,2010,July,617764.26
8,2010,August,663642.07
9,2010,September,844100.37


**Answer:** Without the incomplete December 2011, the monthly net revenue was between about £480,000 and £1.43 million. In both years, revenue rises strongly from September to November, which fits a giftware business before Christmas. The low value for December 2011, at £441,367, is caused by the missing data after 9 December.

#### Q2. Which month had the highest net revenue?

In [43]:
q("""
    WITH revenue AS (
        SELECT
            year(InvoiceDate) AS year,
            month(InvoiceDate) AS month_number,
            monthname(InvoiceDate) AS month,
            SUM(line_total) AS net_revenue
        FROM v_sales_lines
        GROUP BY
            year(InvoiceDate),
            month(InvoiceDate),
            monthname(InvoiceDate)
    )
    SELECT
        year,
        month,
        ROUND(net_revenue, 2) AS net_revenue
    FROM revenue
    ORDER BY net_revenue DESC
    LIMIT 1
""")

,year,month,net_revenue
0,2011,November,1432734.99


**Answer:** In November 2011, net revenue reached a high of £1,432,735, which was only a little bit higher than the £1,400,559 from November 2010.

#### Q3. What was the month-over-month revenue growth?

I exclude December 2011 because it is incomplete.

In [44]:
q("""
    WITH monthly_revenue AS (
        SELECT
            year(InvoiceDate) AS year,
            month(InvoiceDate) AS month_number,
            monthname(InvoiceDate) AS month,
            SUM(line_total) AS net_revenue
        FROM v_sales_lines
        WHERE
            InvoiceDate >= TIMESTAMP '2009-12-01 00:00:00'
            AND InvoiceDate < TIMESTAMP '2011-12-01 00:00:00'
        GROUP BY
            year(InvoiceDate),
            month(InvoiceDate),
            monthname(InvoiceDate)
    ),
    previous_with_last_net_revenue AS (
        SELECT
            *,
            LAG(net_revenue) OVER (ORDER BY
                year,
                month_number
            ) AS previous_net_revenue
        FROM monthly_revenue
    )
    SELECT
        year,
        month,
        net_revenue,
        ROUND(previous_net_revenue, 2) AS previous_month_revenue,
        ROUND(100.0
            * (net_revenue - previous_net_revenue)
            / NULLIF(previous_net_revenue, 0)
            , 2) AS monthly_growth_percent
    FROM previous_with_last_net_revenue
    ORDER BY
        year,
        month_number
""")

,year,month,net_revenue,previous_month_revenue,monthly_growth_percent
0,2009,December,781541.690,NaN,NaN
1,2010,January,606385.182,781541.69,-22.41
2,2010,February,527160.296,606385.18,-13.07
3,2010,March,754676.651,527160.30,43.16
4,2010,April,638968.562,754676.65,-15.33
5,2010,May,608351.250,638968.56,-4.79
6,2010,June,672935.410,608351.25,10.62
7,2010,July,617764.260,672935.41,-8.20
8,2010,August,663642.070,617764.26,7.43
9,2010,September,844100.371,663642.07,27.19


**Answer:** The month-over-month growth varies a lot, from −45.7% in December 2010 to +51.6% in May 2011. The pattern is mainly seasonal: the growth is positive from September to November and strongly negative in December and January. Because of this seasonality, comparing the same months of two years is more meaningful.

#### Q4. What was the running total of net revenue within each year?

I exclude December 2009 because it is the only month of 2009 in the data.

In [45]:
q("""
    WITH monthly_revenue AS (
        SELECT
            year(InvoiceDate) AS year,
            month(InvoiceDate) AS month_number,
            monthname(InvoiceDate) AS month,
            SUM(line_total) AS net_revenue
        FROM v_sales_lines
        WHERE
            InvoiceDate >= TIMESTAMP '2010-01-01 00:00:00'
            AND InvoiceDate < TIMESTAMP '2011-12-31 00:00:00'
        GROUP BY
            year(InvoiceDate),
            month(InvoiceDate),
            monthname(InvoiceDate)
    )
    SELECT
        year,
        month,
        net_revenue,
        SUM(net_revenue) OVER (PARTITION BY year ORDER BY month_number) AS running_total_within_the_year
    FROM monthly_revenue
    ORDER BY
        year,
        month_number
""")

,year,month,net_revenue,running_total_within_the_year
0,2010,January,606385.182,606385.182
1,2010,February,527160.296,1133545.478
2,2010,March,754676.651,1888222.129
3,2010,April,638968.562,2527190.691
4,2010,May,608351.250,3135541.941
5,2010,June,672935.410,3808477.351
6,2010,July,617764.260,4426241.611
7,2010,August,663642.070,5089883.681
8,2010,September,844100.371,5933984.052
9,2010,October,1072467.650,7006451.702


**Answer:** The net revenue for 2010 was £9.17 million. From January to November 2011, the net revenue was £8.59 million, about 2% higher than the £8.41 million for the same period in 2010. The total for 2011, £9.03 million is not complete, because the data ends on 9 December 2011.

### 5.2 Countries

#### Q5. Which countries generate the most net revenue, and what is their share of the total?

In [46]:
q("""
    WITH revenue AS (
        SELECT
            Country,
            SUM(line_total) AS net_revenue
        FROM v_sales_lines
        GROUP BY Country
    )
    SELECT
        Country,
        net_revenue,
        ROUND(100.0
            * net_revenue
            / SUM(net_revenue) OVER ()
            , 2) AS percentage_net_revenue
    FROM revenue
    ORDER BY net_revenue DESC
    LIMIT 10
""")

,Country,net_revenue,percentage_net_revenue
0,United Kingdom,1.622108e+07,85.46
1,EIRE,6.034071e+05,3.18
2,Netherlands,5.460971e+05,2.88
3,Germany,3.751322e+05,1.98
4,France,2.936263e+05,1.55
5,Australia,1.663496e+05,0.88
6,Switzerland,9.294036e+04,0.49
7,Spain,8.486288e+04,0.45
8,Sweden,8.437907e+04,0.44
9,Denmark,6.335559e+04,0.33


**Answer:** The United Kingdom generates 85.46% of the net revenue. It is followed by Ireland at 3.18%, the Netherlands at 2.88%, Germany at 1.98% and France at 1.55%. The business depends strongly on its home market.

#### Q6. What is the average order value and the average number of units per order in each country?

Only countries with at least 100 orders are included. Cancellation invoices are not counted as orders.

In [47]:
q("""
    WITH orders AS (
        SELECT
            Invoice,
            Country,
            SUM(line_total) AS order_value,
            SUM(Quantity) AS units_in_order
        FROM v_sales_lines
        WHERE Invoice NOT LIKE 'C%'
        GROUP BY
            Invoice,
            Country
    )
    SELECT
        Country,
        COUNT(*) AS number_of_orders,
        ROUND(
            AVG(order_value)
            , 2) AS average_order_value,
        ROUND(
            MEDIAN(order_value)
            , 2) AS median_order_value,
        ROUND(
            AVG(units_in_order)
            , 2) AS average_units_per_order,
        ROUND(
            MEDIAN(units_in_order)
            , 2) AS median_units_per_order
    FROM orders
    GROUP BY Country
    HAVING COUNT(*) >= 100
    ORDER BY average_order_value DESC
""")

,Country,number_of_orders,average_order_value,median_order_value,average_units_per_order,median_units_per_order
0,Netherlands,216,2545.25,657.08,1776.05,392.0
1,EIRE,581,1073.66,694.66,578.87,324.0
2,Spain,144,679.29,364.37,347.35,227.0
3,France,598,520.55,360.07,452.81,216.0
4,Germany,753,509.76,360.30,296.55,206.0
5,United Kingdom,36184,465.85,296.76,254.51,144.0
6,Belgium,143,398.69,301.56,240.23,187.0


**Answer:** The Netherlands has by far the highest average order value at £2,545 and the most units per order, with a median of 392. Customers there probably buy for resale. Ireland has the highest median order value, at £695. The United Kingdom has the most orders, at 36,184, but the lowest median order value, at £297. In every country, the average is much higher than the median, which means that a few very large orders pull the average up.

### 5.3 Customers

#### Q7. Who are the 10 customers with the highest net revenue?

The percentages refer to the net revenue of all customers with a customer ID.

In [48]:
q("""
    WITH revenue AS (
        SELECT
            Customer_ID,
            SUM(line_total) AS net_revenue
        FROM v_sales_lines
        WHERE Customer_ID IS NOT NULL
        GROUP BY Customer_ID
    )
    SELECT
        Customer_ID,
        net_revenue,
        ROUND(100.0
            * net_revenue
            / SUM(net_revenue) OVER ()
            , 2) AS percentage_net_revenue
    FROM revenue
    ORDER BY net_revenue DESC
    LIMIT 10

""")

,Customer_ID,net_revenue,percentage_net_revenue
0,18102,578408.64,3.52
1,14646,523202.74,3.19
2,14156,297961.56,1.82
3,14911,258601.47,1.58
4,17450,233803.91,1.42
5,13694,190354.06,1.16
6,17511,168504.44,1.03
7,12415,143107.02,0.87
8,16684,141530.29,0.86
9,15061,124982.13,0.76


**Answer:** Customer 18102 has the highest net revenue, at £578,409, followed by customer 14646 with £523,203. Together, these two customers generate 6.7% of the net revenue of all customers with an ID. Measured against the total net revenue including orders without a customer ID, customer 18102 has a share of 3.05%.

#### Q8. What share of the net revenue do the top 10 customers generate together?

In [49]:
q("""
    WITH revenue AS (
        SELECT
            Customer_ID,
            SUM(line_total) AS net_revenue
        FROM v_sales_lines
        WHERE Customer_ID IS NOT NULL
        GROUP BY Customer_ID
    ),
    previous_with_percentage AS (
        SELECT
            *,
            net_revenue / SUM(net_revenue) OVER () AS percentage_net_revenue
        FROM revenue
        ORDER BY net_revenue DESC
        LIMIT 10
    )
    SELECT
        ROUND(100.0
            * SUM(percentage_net_revenue)
            , 2) AS percentage_net_revenue_top_10_customers
    FROM previous_with_percentage
""")

,percentage_net_revenue_top_10_customers
0,16.21


**Answer:** The top 10 customers generate 16.21% of the net revenue of all customers with an ID. Measured against the total net revenue including orders without a customer ID, their share is about 14.0%. This means the business depends noticeably on a small number of large customers.

#### Q9. How many customers ordered only once or several times, and how much net revenue does each group generate?

Only invoices that are not cancellations count as orders. Customers with nothing but cancellations form a separate group, as their original purchases were made before December 2009.

In [50]:
q("""
    WITH customers AS (
        SELECT
            Customer_ID,
            COUNT(DISTINCT Invoice) FILTER (WHERE Invoice NOT LIKE 'C%') AS orders,
            SUM(line_total) AS net_revenue
        FROM v_sales_lines
        WHERE Customer_ID IS NOT NULL
        GROUP BY Customer_ID
    )
    SELECT
        CASE
            WHEN orders = 1 THEN 'One-time'
            WHEN orders > 1 THEN 'Repeat'
            ELSE 'Only cancellations'
            END AS customer_type,
        COUNT(*) AS customers,
        ROUND(100.0
                * COUNT(*)
                / SUM(COUNT(*)) OVER ()
                , 2) AS percentage_customers,
        ROUND(SUM(net_revenue), 2) AS net_revenue,
        ROUND(100.0
            * SUM(net_revenue)
            / SUM(SUM(net_revenue)) OVER ()
            , 2) AS percentage_net_revenue
    FROM customers
    GROUP BY customer_type
    ORDER BY customers DESC
""")

,customer_type,customers,percentage_customers,net_revenue,percentage_net_revenue
0,Repeat,4234,72.07,15868150.28,96.69
1,One-time,1618,27.54,545150.59,3.32
2,Only cancellations,23,0.39,-1406.13,-0.01


**Answer:** About 72% of the customers ordered more than once, and they generate about 97% of the net revenue of all customers with an ID. The 1,618 one-time customers generate only about 3%. Repeat customers are clearly the core of this business.

#### Q10. How many days pass between two orders of the same customer?

First, I count every invoice as an order. If a customer has several invoices on the same day, this creates gaps of 0 days.

In [51]:
q("""
    WITH main_with_previous_invoicedate AS (
        SELECT
            Customer_ID,
            Invoice,
            MIN(InvoiceDate) AS InvoiceDate
        FROM v_sales_lines
        WHERE
            Customer_ID IS NOT NULL
            AND Invoice NOT LIKE 'C%'
        GROUP BY
            Customer_ID,
            Invoice
    ),
    previous_with_previous_invoicedate AS (
        SELECT
            *,
            LAG(InvoiceDate) OVER (PARTITION BY Customer_ID ORDER BY InvoiceDate) AS previous_invoicedate
        FROM main_with_previous_invoicedate
    ),
    previous_with_days_between_orders AS (
        SELECT
            date_diff('day', previous_invoicedate, InvoiceDate) AS days_between_orders
        FROM previous_with_previous_invoicedate
        WHERE previous_invoicedate IS NOT NULL
    )
    SELECT
        ROUND(
            AVG(days_between_orders)
            , 1) AS average_days_between_orders,
        MEDIAN(days_between_orders) AS median_days_between_orders,
        COUNT(*) AS number_of_gaps
    FROM previous_with_days_between_orders
""")

,average_days_between_orders,median_days_between_orders,number_of_gaps
0,52.1,25.0,30742


Second, I count at maximum one order per customer and day:

In [52]:
q("""
    WITH main_with_previous_invoicedate AS (
        SELECT DISTINCT
            Customer_ID,
            CAST(InvoiceDate AS DATE) AS InvoiceDate
        FROM v_sales_lines
        WHERE
            Customer_ID IS NOT NULL
            AND Invoice NOT LIKE 'C%'
    ),
    previous_with_previous_invoicedate AS (
        SELECT
            *,
            LAG(InvoiceDate) OVER (PARTITION BY Customer_ID ORDER BY InvoiceDate) AS previous_invoicedate
        FROM main_with_previous_invoicedate
    ),
    previous_with_days_between_orders AS (
        SELECT
            date_diff('day', previous_invoicedate, InvoiceDate) AS days_between_orders
        FROM previous_with_previous_invoicedate
        WHERE previous_invoicedate IS NOT NULL
    )
    SELECT
        ROUND(
            AVG(days_between_orders)
            , 1) AS average_days_between_orders,
        MEDIAN(days_between_orders) AS median_days_between_orders,
        COUNT(*) AS number_of_gaps
    FROM previous_with_days_between_orders
""")

,average_days_between_orders,median_days_between_orders,number_of_gaps
0,59.3,32.0,27026


**Answer:** If every invoice counts as an order, the average gap between two orders is 52.1 days and the median is 25 days. If several invoices on the same day count as one order, the average is 59.3 days and the median is 32 days. 3,716 gaps of 0 days are removed in this case. Half of all repeat orders come within about one month, but some customers wait much longer, which pulls the average up.

### 5.4 Products

#### Q11. What are the top 10 products by net revenue?

In [53]:
q("""
    WITH revenue AS (
        SELECT
            StockCode,
            MODE(Description) AS description,
            SUM(line_total) AS net_revenue
        FROM v_sales_lines
        GROUP BY StockCode
    )
    SELECT
        StockCode,
        description,
        ROUND(net_revenue, 2) AS net_revenue
    FROM revenue
    ORDER BY net_revenue DESC
    LIMIT 10
""")

,StockCode,description,net_revenue
0,22423,REGENCY CAKESTAND 3 TIER,314513.47
1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,248678.81
2,85099B,JUMBO BAG RED RETROSPOT,178719.21
3,47566,PARTY BUNTING,147157.43
4,84879,ASSORTED COLOUR BIRD ORNAMENT,128907.01
5,22086,PAPER CHAIN KIT 50'S CHRISTMAS,116422.49
6,79321,CHILLI LIGHTS,80237.48
7,22197,SMALL POPCORN HOLDER,78935.92
8,22386,JUMBO BAG PINK POLKADOT,75518.22
9,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,71031.15


**Answer:** The REGENCY CAKESTAND 3 TIER generates the most net revenue at £314,513. It is followed by the WHITE HANGING HEART T-LIGHT HOLDER at £248,679 and the JUMBO BAG RED RETROSPOT at £178,719. The list mainly contains home decoration, party and Christmas products.

#### Q12. What are the top 10 products by quantity sold?

In [54]:
q("""
    SELECT
        StockCode,
        CAST(
            SUM(Quantity) AS INTEGER
        ) AS quantity,
        MODE(Description) AS description
    FROM v_sales_lines
    GROUP BY StockCode
    ORDER BY quantity DESC
    LIMIT 10
""")

,StockCode,quantity,description
0,84077,104675,WORLD WAR 2 GLIDERS ASSTD DESIGNS
1,85099B,95718,JUMBO BAG RED RETROSPOT
2,21212,93862,PACK OF 72 RETROSPOT CAKE CASES
3,85123A,90686,WHITE HANGING HEART T-LIGHT HOLDER
4,22197,87733,SMALL POPCORN HOLDER
5,84879,79782,ASSORTED COLOUR BIRD ORNAMENT
6,17003,69665,BROCADE RING PURSE
7,21977,56011,PACK OF 60 PINK PAISLEY CAKE CASES
8,84991,53856,60 TEATIME FAIRY CAKE CASES
9,15036,43343,ASSORTED COLOURS SILK FAN


The same query without same-day cancellations:

In [55]:
q("""
    SELECT
        StockCode,
        CAST(
            SUM(Quantity) AS INTEGER
        ) AS quantity,
        MODE(Description) AS description
    FROM v_sales_lines_clean
    GROUP BY StockCode
    ORDER BY quantity DESC
    LIMIT 10
""")

,StockCode,quantity,description
0,84077,104675,WORLD WAR 2 GLIDERS ASSTD DESIGNS
1,85099B,95508,JUMBO BAG RED RETROSPOT
2,21212,93862,PACK OF 72 RETROSPOT CAKE CASES
3,85123A,90674,WHITE HANGING HEART T-LIGHT HOLDER
4,22197,87733,SMALL POPCORN HOLDER
5,84879,79782,ASSORTED COLOUR BIRD ORNAMENT
6,17003,69665,BROCADE RING PURSE
7,21977,55987,PACK OF 60 PINK PAISLEY CAKE CASES
8,84991,53832,60 TEATIME FAIRY CAKE CASES
9,15036,43343,ASSORTED COLOURS SILK FAN


**Answer:** WORLD WAR 2 GLIDERS ASSTD DESIGNS is the product with the most units sold at 104,675. Without same-day cancellations, the ranking stays the same and the quantities change by less than 0.3%.

#### Q13. What are the top 3 products by net revenue in each month?

In [56]:
q("""
    WITH product_revenue AS (
        SELECT
            year(InvoiceDate) AS year,
            month(InvoiceDate) AS month_number,
            monthname(InvoiceDate) AS month,
            StockCode,
            MODE(Description) AS description,
            SUM(line_total) AS net_revenue
        FROM v_sales_lines
        GROUP BY
            year(InvoiceDate),
            month(InvoiceDate),
            monthname(InvoiceDate),
            StockCode
    ),
    ranked AS (
        SELECT
            *,
            RANK() OVER (PARTITION BY
                year,
                month_number
            ORDER BY net_revenue DESC) AS net_revenue_rank
        FROM product_revenue
    )
    SELECT
        year,
        month,
        net_revenue_rank,
        StockCode,
        description,
        net_revenue
    FROM ranked
    WHERE net_revenue_rank <= 3
    ORDER BY
        year,
        month_number,
        net_revenue_rank
""")

,year,month,net_revenue_rank,StockCode,description,net_revenue
0,2009,December,1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,17247.70
1,2009,December,2,22086,PAPER CHAIN KIT 50'S CHRISTMAS,10169.36
2,2009,December,3,15056BL,EDWARDIAN PARASOL BLACK,8713.40
3,2010,January,1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,14643.79
4,2010,January,2,20748,KENSINGTON COFFEE SET,10248.60
5,2010,January,3,85099B,JUMBO BAG RED WHITE SPOTTY,6288.81
6,2010,February,1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,7975.17
7,2010,February,2,20685,DOOR MAT RED SPOT,6399.26
8,2010,February,3,85099B,JUMBO BAG RED WHITE SPOTTY,4733.85
9,2010,March,1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,14151.80


In [57]:
q("""
    WITH product_revenue AS (
        SELECT
            year(InvoiceDate) AS year,
            month(InvoiceDate) AS month_number,
            monthname(InvoiceDate) AS month,
            StockCode,
            MODE(Description) AS description,
            SUM(line_total) AS net_revenue
        FROM v_sales_lines
        GROUP BY
            year(InvoiceDate),
            month(InvoiceDate),
            monthname(InvoiceDate),
            StockCode
    ),
    ranked AS (
        SELECT
            *,
            RANK() OVER (PARTITION BY
                year,
                month_number
            ORDER BY net_revenue DESC) AS net_revenue_rank
        FROM product_revenue
    ),
    product_revenue_with_ranked AS (
        SELECT
            year,
            month,
            net_revenue_rank,
            StockCode,
            description,
            net_revenue
        FROM ranked
        WHERE net_revenue_rank <= 3
        ORDER BY
            year,
            month_number,
            net_revenue_rank
    )
    SELECT
        StockCode,
        description,
        COUNT(*) AS number_of_months
    FROM product_revenue_with_ranked
    GROUP BY 
        StockCode,
        description
    ORDER BY COUNT(*) DESC
    LIMIT 5
""")

,StockCode,description,number_of_months
0,22423,REGENCY CAKESTAND 3 TIER,19
1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,15
2,47566,PARTY BUNTING,9
3,85099B,JUMBO BAG RED RETROSPOT,6
4,22086,PAPER CHAIN KIT 50'S CHRISTMAS,5


**Answer:** The REGENCY CAKESTAND 3 TIER is in the top 3 in 19 of 25 months, and the WHITE HANGING HEART T-LIGHT HOLDER in 15 of 25 months.

### 5.5 Order times

#### Q14. On which weekdays do the most orders come in?

In [58]:
q("""
    WITH day_of_week AS (
        SELECT
            isodow(InvoiceDate) AS weekday_number,
            dayname(InvoiceDate) AS weekday,
            COUNT(DISTINCT Invoice) AS orders
        FROM v_sales_lines
        WHERE
            Invoice NOT LIKE 'C%'
            AND Quantity > 0
        GROUP BY
            isodow(InvoiceDate),
            dayname(InvoiceDate)
    )
    SELECT
        weekday_number,
        weekday,
        orders,
        ROUND(100.0
            * orders
            / SUM(orders) OVER ()
            , 2) AS percent_orders
    FROM day_of_week
    ORDER BY orders DESC
""")

,weekday_number,weekday,orders,percent_orders
0,4,Thursday,8167,20.67
1,2,Tuesday,7147,18.09
2,3,Wednesday,7115,18.01
3,1,Monday,6245,15.80
4,5,Friday,5988,15.15
5,7,Sunday,4824,12.21
6,6,Saturday,30,0.08


**Answer:** Thursday has the most orders at 20.7%. It is followed by Tuesday and Wednesday, at 18% each. There are almost no orders on Saturdays, only 30 in two years, so the business is probably closed on Saturdays.

#### Q15. At which hours do the most orders come in?

In [59]:
q("""
    WITH hours AS (
        SELECT
            hour(InvoiceDate) AS hour,
            COUNT(DISTINCT Invoice) AS orders
        FROM v_sales_lines
        WHERE
            Invoice NOT LIKE 'C%'
            AND Quantity > 0
        GROUP BY hour(InvoiceDate)
    )
    SELECT
        hour,
        orders,
        ROUND(100.0
            * orders
            / SUM(orders) OVER ()
            , 2) AS percent_orders
    FROM hours
    ORDER BY orders DESC
""")

,hour,orders,percent_orders
0,12,6369,16.12
1,13,5638,14.27
2,14,4974,12.59
3,11,4837,12.24
4,10,4563,11.55
5,15,4487,11.35
6,9,2720,6.88
7,16,2628,6.65
8,17,1381,3.49
9,8,995,2.52


**Answer:** 78% of all orders come in between 10:00 and 16:00. The busiest hour is 12:00–13:00 with 16.1% of all orders. Very few orders come in before 8:00 or after 19:00.

#### Q16. At which combination of weekday and hour do the most orders come in?

In [60]:
q("""
    WITH weekday_orders AS (
        SELECT
            isodow(InvoiceDate) AS weekday_number,
            dayname(InvoiceDate) AS weekday,
            hour(InvoiceDate) AS hour,
            COUNT(DISTINCT Invoice) AS orders
        FROM v_sales_lines
        WHERE
            Invoice NOT LIKE 'C%'
            AND Quantity > 0
        GROUP BY
            isodow(InvoiceDate),
            dayname(InvoiceDate),
            hour(InvoiceDate)
    )
    SELECT
        weekday_number,
        weekday,
        hour,
        orders,
        ROUND(100.0
            * orders
            / SUM(orders) OVER ()
            , 2) AS percent_orders
    FROM weekday_orders
    ORDER BY orders DESC
    LIMIT 5
""")

,weekday_number,weekday,hour,orders,percent_orders
0,3,Wednesday,12,1212,3.07
1,4,Thursday,12,1138,2.88
2,2,Tuesday,12,1109,2.81
3,3,Wednesday,13,1041,2.63
4,2,Tuesday,13,1038,2.63


**Answer:** The busiest times are around lunchtime from Tuesday to Thursday. The busiest single hour is Wednesday 12:00–13:00 with 3.07% of all orders.

### 5.6 Cancellations and returns

The dataset does not contain returns directly. Invoices that start with `C` are cancellations. In this section I use these cancellations as an approximation for returns.

#### Q17. What share of the sales value and quantity is cancelled per month?

In [61]:
q("""
    WITH monthly AS (
        SELECT
            year(InvoiceDate) AS year,
            month(InvoiceDate) AS month_number,
            monthname(InvoiceDate) AS month,
            SUM(line_total) FILTER (WHERE Invoice NOT LIKE 'C%') AS sales_revenue,
            -SUM(line_total) FILTER (WHERE Invoice LIKE 'C%') AS returned_revenue,
            SUM(Quantity) FILTER (WHERE Invoice NOT LIKE 'C%') AS sales_quantity,
            -SUM(Quantity) FILTER (WHERE Invoice LIKE 'C%') AS returned_quantity
        FROM v_sales_lines
        GROUP BY
            year(InvoiceDate),
            month(InvoiceDate),
            monthname(InvoiceDate)
    )
    SELECT
        year,
        month,
        sales_revenue,
        returned_revenue,
        ROUND(100.0
            * returned_revenue
            / NULLIF(sales_revenue, 0)
            , 2) AS percent_revenue_returned,
        sales_quantity,
        returned_quantity,
        ROUND(100.0
            * returned_quantity
            / NULLIF(sales_quantity, 0)
            , 2) AS percent_quantity_returned
    FROM monthly
    ORDER BY
        year,
        month_number
""")

,year,month,sales_revenue,returned_revenue,percent_revenue_returned,sales_quantity,returned_quantity,percent_quantity_returned
0,2009,December,801102.970,19561.28,2.44,426738.0,9984.0,2.34
1,2010,January,613850.812,7465.63,1.22,391267.0,3304.0,0.84
2,2010,February,539744.686,12584.39,2.33,382528.0,6740.0,1.76
3,2010,March,764333.851,9657.20,1.26,526362.0,4274.0,0.81
4,2010,April,648993.822,10025.26,1.54,367559.0,5998.0,1.63
5,2010,May,645645.390,37294.14,5.78,396776.0,16568.0,4.18
6,2010,June,699316.020,26380.61,3.77,408054.0,15367.0,3.77
7,2010,July,634896.990,17132.73,2.70,338581.0,5987.0,1.77
8,2010,August,676060.720,12418.65,1.84,472971.0,6276.0,1.33
9,2010,September,871878.341,27777.97,3.19,584855.0,94286.0,16.12


The same query without same-day cancellations:

In [62]:
q("""
    WITH monthly AS (
        SELECT
            year(InvoiceDate) AS year,
            month(InvoiceDate) AS month_number,
            monthname(InvoiceDate) AS month,
            SUM(line_total) FILTER (WHERE Invoice NOT LIKE 'C%') AS sales_revenue,
            -SUM(line_total) FILTER (WHERE Invoice LIKE 'C%') AS returned_revenue,
            SUM(Quantity) FILTER (WHERE Invoice NOT LIKE 'C%') AS sales_quantity,
            -SUM(Quantity) FILTER (WHERE Invoice LIKE 'C%') AS returned_quantity
        FROM v_sales_lines_clean
        GROUP BY
            year(InvoiceDate),
            month(InvoiceDate),
            monthname(InvoiceDate)
    )
    SELECT
        year,
        month,
        sales_revenue,
        returned_revenue,
        ROUND(100.0
            * returned_revenue
            / NULLIF(sales_revenue, 0)
            , 2) AS percent_revenue_returned,
        sales_quantity,
        returned_quantity,
        ROUND(100.0
            * returned_quantity
            / NULLIF(sales_quantity, 0)
            , 2) AS percent_quantity_returned
    FROM monthly
    ORDER BY
        year,
        month_number
""")

,year,month,sales_revenue,returned_revenue,percent_revenue_returned,sales_quantity,returned_quantity,percent_quantity_returned
0,2009,December,797675.620,16393.01,2.06,424480.0,7998.0,1.88
1,2010,January,613463.182,7065.50,1.15,391089.0,3115.0,0.80
2,2010,February,539004.546,11868.31,2.20,381978.0,6284.0,1.65
3,2010,March,762859.261,8314.01,1.09,525324.0,3327.0,0.63
4,2010,April,645929.492,7967.83,1.23,366439.0,5078.0,1.39
5,2010,May,630582.710,22367.06,3.55,384686.0,5684.0,1.48
6,2010,June,681848.330,12501.42,1.83,394544.0,4367.0,1.11
7,2010,July,625175.240,10484.29,1.68,336793.0,4476.0,1.33
8,2010,August,675079.390,11457.12,1.70,471854.0,5171.0,1.10
9,2010,September,862946.341,22469.30,2.60,579985.0,91383.0,15.76


**Answer:** In most months, between 1% and 4% of the sales value is cancelled. There are three large spikes in January 2011, December 2011 and September 2010. Without same-day cancellations, the spikes in January and December 2011 disappear, and the cancelled share of the value stays below 4% in every month. The spike in September 2010 remains, so these cancellations did not happen on the same day as the order.

#### Q18. Which 10 products have the highest return rate?

Return rate = cancelled units / sold units. I only include products with at least 100 sold units, so that products with very few sales do not distort the ranking.

In [63]:
q("""
    WITH sales AS (
        SELECT
            StockCode,
            MODE(Description) AS description,
            SUM(Quantity) FILTER (WHERE Invoice NOT LIKE 'C%') AS sold
        FROM v_sales_lines
        GROUP BY StockCode
    ),
    returned AS (
        SELECT
            StockCode,
            MODE(Description) AS description,
            -SUM(Quantity) FILTER (WHERE Invoice LIKE 'C%') AS returned
        FROM v_sales_lines
        GROUP BY StockCode
    )
    SELECT
        s.StockCode,
        s.description,
        CAST(sold AS INTEGER) AS sold,
        CAST(returned AS INTEGER) AS returned,
        ROUND(100.0
            * returned
            / NULLIF(sold, 0)
            , 2) AS percentage_return_rate
    FROM sales s
    INNER JOIN returned r ON s.StockCode = r.StockCode
    WHERE sold >= 100
    ORDER BY percentage_return_rate DESC
    LIMIT 10
""")

,StockCode,description,sold,returned,percentage_return_rate
0,79323B,BLACK CHERRY LIGHTS,243,248,102.06
1,20715,LITTLE FLOWER SHOPPER BAG,400,400,100.00
2,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,80995,100.00
3,23166,MEDIUM CERAMIC TOP STORAGE JAR,78033,74494,95.46
4,72045D,ROSES ON BLUE TEACUP CANDLE,539,504,93.51
5,21950,SET OF 6 DOTS CHOPSTICKS,211,181,85.78
6,84925D,LA PALMIERA WALL THERMOMETER,452,372,82.30
7,23113,PANTRY CHOPPING BOARD,1154,946,81.98
8,20952,PORCELAIN CHERUB BELL SMALL,330,260,78.79
9,10138,ASSORTED COLOUR JUMBO PEN,261,194,74.33


The same query without same-day cancellations:

In [64]:
q("""
    WITH sales AS (
        SELECT
            StockCode,
            MODE(Description) AS description,
            SUM(Quantity) FILTER (WHERE Invoice NOT LIKE 'C%') AS sold
        FROM v_sales_lines_clean
        GROUP BY StockCode
    ),
    returned AS (
        SELECT
            StockCode,
            MODE(Description) AS description,
            -SUM(Quantity) FILTER (WHERE Invoice LIKE 'C%') AS returned
        FROM v_sales_lines_clean
        GROUP BY StockCode
    )
    SELECT
        s.StockCode,
        s.description,
        CAST(sold AS INTEGER) AS sold,
        CAST(returned AS INTEGER) AS returned,
        ROUND(100.0
            * returned
            / NULLIF(sold, 0)
            , 2) AS percentage_return_rate
    FROM sales s
    INNER JOIN returned r ON s.StockCode = r.StockCode
    WHERE sold >= 100
    ORDER BY percentage_return_rate DESC
    LIMIT 10
""")

,StockCode,description,sold,returned,percentage_return_rate
0,79323B,BLACK CHERRY LIGHTS,221,226,102.26
1,20715,LITTLE FLOWER SHOPPER BAG,400,400,100.00
2,21950,SET OF 6 DOTS CHOPSTICKS,211,181,85.78
3,84925D,LA PALMIERA WALL THERMOMETER,452,372,82.30
4,23113,PANTRY CHOPPING BOARD,1154,946,81.98
5,10138,ASSORTED COLOUR JUMBO PEN,261,194,74.33
6,23055,IVORY CHANDELIER T-LIGHT HOLDER,418,288,68.90
7,23056,FLOWERS CHANDELIER T-LIGHT HOLDER,438,291,66.44
8,79341,WILLOW BRANCH LIGHTS.,725,448,61.79
9,79323G,GOLD CHERRY LIGHTS,298,179,60.07


**Answer:** With all cancellations, the ranking is distorted by two huge orders that were cancelled on the same day. These were PAPER CRAFT, LITTLE BIRDIE and MEDIUM CERAMIC TOP STORAGE JAR. Without same-day cancellations, both products disappear from the list. BLACK CHERRY LIGHTS still has a return rate above 100%. This is possible if units were bought before December 2009, when the data starts, and cancelled later. Products with high return rates, such as SET OF 6 DOTS CHOPSTICKS, LA PALMIERA WALL THERMOMETER and PANTRY CHOPPING BOARD, could be checked for quality problems.

#### Q19. What share of all sold units is cancelled?

In [65]:
q("""
    WITH sales AS (
        SELECT
            SUM(Quantity) AS sold
        FROM v_sales_lines
        WHERE Invoice NOT LIKE 'C%'
    ),
    returned AS (
        SELECT
        -SUM(Quantity) AS returned
        FROM v_sales_lines
        WHERE Invoice LIKE 'C%'
    )
   SELECT
        ROUND(100
            * CAST(r.returned AS DOUBLE)
            / NULLIF(s.sold, 0)
            , 2) AS percentage_overall_return_rate
   FROM sales s
   CROSS JOIN returned r
""")

,percentage_overall_return_rate
0,4.19


The same query without same-day cancellations:

In [66]:
q("""
    WITH sales AS (
        SELECT
            SUM(Quantity) AS sold
        FROM v_sales_lines_clean
        WHERE Invoice NOT LIKE 'C%'
    ),
    returned AS (
        SELECT
        -SUM(Quantity) AS returned
        FROM v_sales_lines_clean
        WHERE Invoice LIKE 'C%'
    )
   SELECT
        ROUND(100
            * CAST(r.returned AS DOUBLE)
            / NULLIF(s.sold, 0)
            , 2) AS percentage_overall_return_rate
   FROM sales s
   CROSS JOIN returned r
""")

,percentage_overall_return_rate
0,2.15


**Answer:** 4.19% of all sold units are cancelled. Without same-day cancellations, the rate is only 2.15%. About half of all cancelled units therefore come from orders that were cancelled on the same day, probably to correct mistakes.

#### Why can a return rate be higher than 100%?

To understand this, I look at two examples: TREE OF NOAH FESTIVE SCENTED CANDLE and PAPER CRAFT, LITTLE BIRDIE with the stock codes 20879, 23843.

In [67]:
q("""
    FROM v_sales_lines
    WHERE StockCode IN ('20879', '23843')
    ORDER BY StockCode
""")

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country,non_product_type,line_total
0,490361,20879,TREE OF NOAH FESTIVE SCENTED CANDLE,2,2009-12-04 17:18:00,2.55,NaN,United Kingdom,None,5.10
1,491425,20879,TREE OF NOAH FESTIVE SCENTED CANDLE,6,2009-12-10 20:04:00,0.65,16905,United Kingdom,None,3.90
2,491425,20879,TREE OF NOAH FESTIVE SCENTED CANDLE,6,2009-12-10 20:04:00,0.65,16905,United Kingdom,None,3.90
3,C491593,20879,TREE OF NOAH FESTIVE SCENTED CANDLE,-96,2009-12-11 13:11:00,2.10,17681,United Kingdom,None,-201.60
4,491636,20879,TREE OF NOAH FESTIVE SCENTED CANDLE,6,2009-12-11 15:52:00,0.65,17754,United Kingdom,None,3.90
5,491656,20879,TREE OF NOAH FESTIVE SCENTED CANDLE,2,2009-12-13 10:59:00,0.65,16719,United Kingdom,None,1.30
6,491662,20879,TREE OF NOAH FESTIVE SCENTED CANDLE,6,2009-12-13 11:16:00,0.65,15169,United Kingdom,None,3.90
7,491671,20879,TREE OF NOAH FESTIVE SCENTED CANDLE,6,2009-12-13 11:46:00,0.65,14161,United Kingdom,None,3.90
8,491692,20879,TREE OF NOAH FESTIVE SCENTED CANDLE,12,2009-12-13 13:21:00,0.65,14514,United Kingdom,None,7.80
9,491959,20879,TREE OF NOAH FESTIVE SCENTED CANDLE,6,2009-12-14 16:11:00,0.65,16064,United Kingdom,None,3.90


- **20879:** On 11 December 2009, 96 units were cancelled. This is more than all sales of this product in the data, which adds up to 69 units. The cancelled units were probably bought before December 2009, so the sale is not in the data. The product has fewer than 100 sold units, so it does not appear in the Q18 ranking.
- **23843:** One customer ordered 80,995 units on 9 December 2011 and cancelled the order 12 minutes later. This was most likely an input error.

Return rates above 100% are therefore caused by the limited time period of the data and by orders that were corrected on the same day.

## 6. Recommendations and limitations

### Recommendations

- **Prepare early for the peak season:** Plan stock, staff and marketing before September, because revenue rises strongly from September to November.
- **Grow outside the UK:** 85% of the revenue comes from the UK. Markets with large orders, such as the Netherlands and Ireland, could be a starting point for growth.
- **Look after the most important customers:** Repeat customers generate about 97% of the revenue from customers with an ID, and the top 10 alone generate 16%. Losing one of them would have a clear effect on revenue.
- **Reduce order errors:** About half of all cancelled units are same-day cancellations, probably corrections of mistakes. A check during ordering could prevent some of them.
- **Check products with high return rates:** Products such as SET OF 6 DOTS CHOPSTICKS, LA PALMIERA WALL THERMOMETER and PANTRY CHOPPING BOARD could be checked for quality problems.

### Limitations

- **Missing customer IDs:** 13.6% of the revenue comes from invoices without a customer ID. The customer analyses for Q7 to Q10 only cover customers with an ID.
- **Cancellations instead of returns:** The data only shows cancellations, not the reasons for them. Cancellations of purchases made before December 2009 cannot be matched to a sale.
- **Incomplete December 2011:** The data ends on 9 December 2011, so this month is not comparable with other months.
- **Duplicated rows:** I kept about 12,000 duplicated rows, because it is not clear whether they are errors. They make up 0.2 to 0.3% of the revenue. 
- **Revenue only:** The data has no costs, so it is not possible to analyze profit.